In [14]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [15]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [16]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator

NUM_QUBITS = 10

def qrng(n):
    # generate random bits using quantum superposition
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    qc.measure(range(n), range(n))

    sim = BasicSimulator()
    result = sim.run(transpile(qc, sim), shots=1).result()

    # reverse bitstring so it matches standard 0 to n-1 indexing
    bits = list(result.get_counts().keys())[0][::-1]
    return [int(b) for b in bits]

#Alice prepares her qubits
print("Alice is preparing the qubits...")
alice_bits = qrng(NUM_QUBITS)
alice_bases = qrng(NUM_QUBITS) # 0 is Z-basis, 1 is X-basis

encoded_qubits = []
for i in range(NUM_QUBITS):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1:
        qc.x(0)
    if alice_bases[i] == 1:
        qc.h(0)
    encoded_qubits.append(qc)

#Bob measures the qubits
print("Bob is measuring the received qubits...")
bob_bases = qrng(NUM_QUBITS)
bob_results = []

sim = BasicSimulator()
for i in range(NUM_QUBITS):
    qc = encoded_qubits[i]

    # Apply H gate if Bob decides to measure in the X-basis
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)

    res = sim.run(transpile(qc, sim), shots=1).result()
    measured_val = int(list(res.get_counts().keys())[0])
    bob_results.append(measured_val)

#Sifting the keys
print("Comparing bases over public channel...")
alice_key = []
bob_key = []

for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_results[i])

print(f"Sifted key length: {len(alice_key)}")

#Security check
# Use the first half of the sifted key to check for interference
sample_size = len(alice_key) // 2
errors = 0

for i in range(sample_size):
    if alice_key[i] != bob_key[i]:
        errors += 1

error_rate = errors / sample_size if sample_size > 0 else 0
print(f"Error rate on tested subset: {error_rate * 100}%")

if error_rate == 0.0:
    print("No eavesdropper detected. Key is secure.")
    # The final key is the remaining, untested half
    final_key = alice_key[sample_size:]
    print(f"Final Shared Key: {final_key}")
else:
    print("Eavesdropper detected! Protocol aborted.")

Alice is preparing the qubits...
Bob is measuring the received qubits...
Comparing bases over public channel...
Sifted key length: 5
Error rate on tested subset: 0.0%
No eavesdropper detected. Key is secure.
Final Shared Key: [0, 0, 1]
